# MATH 5110 Lab 2b Study File  
## Change of Coordinates for Circular Orbit

**Purpose.** This independent-study notebook explains how to compute circular orbital motion using a change of coordinates.

The main idea is:

> Rotation about a tilted axis is hard in the standard basis, but easy in a basis adapted to the orbit.

We use two bases:

- $E=\{e_1,e_2,e_3\}$, the standard basis.
- $U=\{P,Q,W\}$, the orbit basis.

Here $W$ is perpendicular to the orbital plane, while $P$ and $Q$ span the orbital plane.

## 1. Rotation in the plane and in space

In the $xy$-plane, counterclockwise rotation by angle $\theta$ has matrix

$$
R_2(\theta)=
\begin{bmatrix}
\cos\theta & -\sin\theta\\
\sin\theta & \cos\theta
\end{bmatrix}.
$$

In three dimensions, rotation about the $z$-axis is

$$
R_3(\theta)=
\begin{bmatrix}
\cos\theta & -\sin\theta & 0\\
\sin\theta & \cos\theta & 0\\
0&0&1
\end{bmatrix}.
$$

This keeps the $z$-coordinate fixed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

def rotation_z(theta):
    return np.array([
        [math.cos(theta), -math.sin(theta), 0],
        [math.sin(theta),  math.cos(theta), 0],
        [0,                0,               1]
    ])

theta = 1.0
print(rotation_z(theta))

## 2. Non-equatorial orbit data

For the non-equatorial orbit in the lab, the normal vector to the orbital plane is

$$
W=\frac13
\begin{bmatrix}
-1\\-2\\2
\end{bmatrix}.
$$

The current position is

$$
x(0)=117.67
\begin{bmatrix}
4\\1\\3
\end{bmatrix}.
$$

The angular velocity is

$$
\omega=3.91\text{ rad/hr}.
$$

At time $t$, the rotation angle is

$$
\theta=\omega t.
$$

In [ ]:
W = np.array([-1, -2, 2], dtype=float)/3
x0 = 117.67*np.array([4, 1, 3], dtype=float)
omega = 3.91

print("W =", W)
print("||W|| =", np.linalg.norm(W))
print("x(0) =", x0)
print("||x(0)|| =", np.linalg.norm(x0))
print("W dot x(0) =", np.dot(W, x0))

The vector $x(0)$ lies in the orbital plane, so it should be perpendicular to $W$.

The computation gives

$$
W\cdot x(0)\approx 0.
$$

## 3. Task 1: Compute $P$ and $Q$

The vector $P$ should be parallel to $x(0)$ and have length $1$:

$$
P=\frac{x(0)}{\|x(0)\|}.
$$

Then

$$
Q=W\times P.
$$

Since $W$ and $P$ are unit and perpendicular, $Q$ is also unit. The ordered basis

$$
U=\{P,Q,W\}
$$

is right-handed.

In [ ]:
P = x0 / np.linalg.norm(x0)
Q = np.cross(W, P)

print("P =", P)
print("Q =", Q)
print("W =", W)

print("||P|| =", np.linalg.norm(P))
print("||Q|| =", np.linalg.norm(Q))
print("||W|| =", np.linalg.norm(W))

print("P dot Q =", np.dot(P,Q))
print("P dot W =", np.dot(P,W))
print("Q dot W =", np.dot(Q,W))
print("W cross P =", np.cross(W,P))

Numerically,

$$
P\approx
\begin{bmatrix}
0.7845\\0.1961\\0.5883
\end{bmatrix},
\qquad
Q\approx
\begin{bmatrix}
-0.5230\\0.7191\\0.4576
\end{bmatrix},
\qquad
W=
\begin{bmatrix}
-0.3333\\-0.6667\\0.6667
\end{bmatrix}.
$$

## 4. Task 2: Change-of-basis matrices

The matrix

$$
[\operatorname{Id}]_{E\leftarrow U}
$$

changes $U$-coordinates into standard $E$-coordinates. Its columns are the $U$-basis vectors written in standard coordinates:

$$
[\operatorname{Id}]_{E\leftarrow U}
=
\begin{bmatrix}
|&|&|\\
P&Q&W\\
|&|&|
\end{bmatrix}.
$$

Since $P,Q,W$ are orthonormal,

$$
[\operatorname{Id}]_{U\leftarrow E}
=
[\operatorname{Id}]_{E\leftarrow U}^{-1}
=
[\operatorname{Id}]_{E\leftarrow U}^{T}.
$$

In [ ]:
Id_EU = np.column_stack([P, Q, W])
Id_UE = np.linalg.inv(Id_EU)

print("[Id]_{E<-U} =")
print(Id_EU)

print("\n[Id]_{U<-E} =")
print(Id_UE)

print("\nTranspose of [Id]_{E<-U} =")
print(Id_EU.T)

print("\nDifference inverse - transpose:")
print(Id_UE - Id_EU.T)

## 5. Why change coordinates?

In the $U$-basis, rotation in the orbital plane is just

$$
[R_c^3]_{U\leftarrow U}
=
\begin{bmatrix}
\cos\theta & -\sin\theta & 0\\
\sin\theta & \cos\theta & 0\\
0&0&1
\end{bmatrix}.
$$

This rotates the $P,Q$ components and keeps the $W$ component fixed.

To get the standard-basis matrix, conjugate:

$$
[R_c^3]_{E\leftarrow E}
=
[\operatorname{Id}]_{E\leftarrow U}
[R_c^3]_{U\leftarrow U}
[\operatorname{Id}]_{U\leftarrow E}.
$$

Then

$$
[x(t)]_E=[R_c^3]_{E\leftarrow E}[x(0)]_E.
$$

In [ ]:
def rotation_U(theta):
    return np.array([
        [math.cos(theta), -math.sin(theta), 0],
        [math.sin(theta),  math.cos(theta), 0],
        [0,                0,               1]
    ])

def rotation_about_W_in_E(theta):
    return Id_EU @ rotation_U(theta) @ Id_UE

def position(t):
    theta = omega*t
    return rotation_about_W_in_E(theta) @ x0

for t in [0.5, 1.0, 1.5]:
    xt = position(t)
    print(f"t = {t} hours")
    print("theta =", omega*t)
    print("x(t) =", xt)
    print("||x(t)|| =", np.linalg.norm(xt))
    print("U-coordinates of x(t) =", Id_UE @ xt)
    print()

## 6. Task 3: Positions at $t=0.5,1,1.5$ hours

Using the above method:

$$
x(0.5)\approx
\begin{bmatrix}
-467.33\\
355.90\\
122.23
\end{bmatrix},
$$

$$
x(1)\approx
\begin{bmatrix}
-120.35\\
-384.47\\
-444.64
\end{bmatrix},
$$

$$
x(1.5)\approx
\begin{bmatrix}
557.55\\
-67.69\\
211.09
\end{bmatrix}.
$$

The length is essentially unchanged:

$$
\|x(t)\|\approx 600.
$$

This is expected because rotations preserve distance from the origin.

## 7. Plot the orbit

The orbit is a circle in the tilted plane spanned by $P$ and $Q$.

Since

$$
x(0)=\|x(0)\|P,
$$

we can parametrize the whole orbit by

$$
x(t)=\|x(0)\|\left(\cos(\omega t)P+\sin(\omega t)Q\right).
$$

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

ts = np.linspace(0, 2*np.pi/omega, 400)
orbit = np.array([position(t) for t in ts])

fig = plt.figure(figsize=(8,7))
ax = fig.add_subplot(111, projection="3d")

ax.plot(orbit[:,0], orbit[:,1], orbit[:,2], label="orbit")
ax.scatter([x0[0]], [x0[1]], [x0[2]], s=60, label="x(0)")

for t in [0.5, 1.0, 1.5]:
    xt = position(t)
    ax.scatter([xt[0]], [xt[1]], [xt[2]], s=60)
    ax.text(xt[0], xt[1], xt[2], f"t={t}")

ax.quiver(0,0,0, 300*W[0],300*W[1],300*W[2], length=1, normalize=False)
ax.text(300*W[0],300*W[1],300*W[2], "W")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Non-equatorial Circular Orbit")
ax.legend()
plt.show()

## 8. Independent-study checkpoints

### Checkpoint A
Why is $Q=W\times P$ perpendicular to both $W$ and $P$?

**Answer.** The cross product of two vectors is perpendicular to both vectors. Since $W$ and $P$ are unit and perpendicular, $Q$ also has length $1$.

### Checkpoint B
Why is $[\operatorname{Id}]_{U\leftarrow E}$ the transpose of $[\operatorname{Id}]_{E\leftarrow U}$?

**Answer.** Because the columns $P,Q,W$ form an orthonormal basis. A matrix with orthonormal columns is orthogonal, so its inverse is its transpose.

### Checkpoint C
Why is the $W$ coordinate unchanged under the rotation matrix in the $U$-basis?

**Answer.** The rotation is around the $W$ direction. Points move in the plane perpendicular to $W$, but the component in the $W$ direction is fixed.

## 9. Summary of key ideas

1. Choose a basis adapted to the geometry:
   $$
   U=\{P,Q,W\}.
   $$

2. Rotation is simple in the adapted basis:
   $$
   \begin{bmatrix}
   \cos\theta&-\sin\theta&0\\
   \sin\theta&\cos\theta&0\\
   0&0&1
   \end{bmatrix}.
   $$

3. Convert back to the standard basis by conjugation:
   $$
   [R]_{E\leftarrow E}
   =
   [\operatorname{Id}]_{E\leftarrow U}
   [R]_{U\leftarrow U}
   [\operatorname{Id}]_{U\leftarrow E}.
   $$

4. Rotations preserve length, so the satellite stays on the circular orbit.